In [1]:
pwd

'/Users/adelumirbaeva/Library/CloudStorage/Dropbox/NU/VGT/files'

In [3]:
!pip install spectral-cube

ERROR: Could not find a version that satisfies the requirement spectral-cube (from versions: none)
ERROR: No matching distribution found for spectral-cube


In [1]:
from pathlib import Path

import astropy.units as u
from spectral_cube import SpectralCube


def split_spectral_cube(
    input_fits,
    ranges,
    output_dir="subcubes",
    prefix=None,
    rest_frequency=None,
    velocity_convention="radio",
    overwrite=True,
):
    """
    Разрезает спектральный FITS-куб по скорости или частоте.

    Parameters
    ----------
    input_fits : str или Path
        Путь к исходному FITS-кубу.

    ranges : list[tuple[Quantity, Quantity]]
        Диапазоны скорости или частоты.

        Пример по скорости:
            [
                (-70*u.km/u.s, -60*u.km/u.s),
                (-60*u.km/u.s, -50*u.km/u.s),
            ]

        Пример по частоте:
            [
                (110.0*u.GHz, 110.1*u.GHz),
                (110.1*u.GHz, 110.2*u.GHz),
            ]

    output_dir : str или Path
        Папка для сабкубов.

    prefix : str, optional
        Начало названий выходных файлов.
        По умолчанию используется имя исходного файла.

    rest_frequency : Quantity, optional
        Частота покоя линии, например 110.201354*u.GHz.
        Нужна при переводе частоты в скорость.

    velocity_convention : {"radio", "optical", "relativistic"}
        Определение доплеровской скорости.

    overwrite : bool
        Перезаписывать существующие файлы.

    Returns
    -------
    list[Path]
        Пути к созданным сабкубам.
    """

    input_fits = Path(input_fits)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if prefix is None:
        prefix = input_fits.stem

    cube = SpectralCube.read(input_fits)
    output_files = []

    for number, limits in enumerate(ranges, start=1):
        if len(limits) != 2:
            raise ValueError(
                f"Диапазон №{number} должен содержать две границы"
            )

        lower, upper = limits

        if not isinstance(lower, u.Quantity):
            raise TypeError(
                f"Нижняя граница диапазона №{number} должна иметь единицу"
            )

        if not isinstance(upper, u.Quantity):
            raise TypeError(
                f"Верхняя граница диапазона №{number} должна иметь единицу"
            )

        if not lower.unit.is_equivalent(upper.unit):
            raise u.UnitConversionError(
                f"Несовместимые единицы: {lower.unit} и {upper.unit}"
            )

        # Приводим границы к одной единице
        upper = upper.to(lower.unit)

        # spectral_slab принимает границы от меньшего значения к большему
        spectral_min = min(lower, upper)
        spectral_max = max(lower, upper)

        if lower.unit.is_equivalent(u.m/u.s):
            # Режем по скорости
            kwargs = {
                "unit": lower.unit,
                "velocity_convention": velocity_convention,
            }

            if rest_frequency is not None:
                kwargs["rest_value"] = rest_frequency

            converted_cube = cube.with_spectral_unit(**kwargs)
            label = "velocity"

        elif lower.unit.is_equivalent(u.Hz):
            # Режем по частоте
            converted_cube = cube.with_spectral_unit(
                lower.unit,
                velocity_convention=velocity_convention,
                rest_value=rest_frequency,
            )
            label = "frequency"

        else:
            raise u.UnitConversionError(
                f"Поддерживаются скорость и частота, получено: {lower.unit}"
            )

        subcube = converted_cube.spectral_slab(
            spectral_min,
            spectral_max,
        )

        if subcube.shape[0] == 0:
            raise ValueError(
                f"В диапазоне {spectral_min} — {spectral_max} "
                "нет спектральных каналов"
            )

        unit_name = lower.unit.to_string().replace("/", "_per_")
        value1 = f"{spectral_min.value:g}".replace("-", "m").replace(".", "p")
        value2 = f"{spectral_max.value:g}".replace("-", "m").replace(".", "p")

        output_path = output_dir / (
            f"{prefix}_{label}_{value1}_to_{value2}_{unit_name}.fits"
        )

        subcube.write(output_path, overwrite=overwrite)
        output_files.append(output_path)

        print(
            f"Сохранён {output_path}: "
            f"{subcube.shape[0]} каналов, "
            f"{subcube.spectral_axis[0]:.5g} — "
            f"{subcube.spectral_axis[-1]:.5g}"
        )

    return output_files

ModuleNotFoundError: No module named 'spectral_cube'